In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from transformers import ViTForImageClassification
import torch.optim as optim
from sklearn.metrics import accuracy_score

In [ ]:
dataset_folder = '/content/drive/MyDrive/Tumor_Trace/Dataset'

In [ ]:
labels = [x[0] for x in os.walk(dataset_folder)][1:]

In [ ]:
labels = [x.split('/')[-1] for x in labels]

In [ ]:
labels

['normal', 'benign', 'malignant']

In [ ]:
data = dict()

In [ ]:
for i in labels:
  data[i] = os.listdir(dataset_folder + '/' + i)

In [ ]:
df = pd.DataFrame([(value, key) for key, values in data.items() for value in values], columns=['image_name', 'labels'])



In [ ]:
df = df.sample(frac=1).reset_index(drop=True)

In [ ]:
df.head()


,image_name,labels
0,malignant (30).png,malignant
1,benign (140)_mask.png,benign
2,normal (17).png,normal
3,malignant (74).png,malignant
4,benign (433)_mask.png,benign


In [ ]:
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['labels'])

In [ ]:
df.head()

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, dir, img, transform=None, is_test=False):
    self.dir = dir
    self.img = img
    self.labels = {0: 'benign', 1:'malignant', 2:'normal'}
    self.transform = transform
    self.is_test = is_test

  def __len__(self):
    return len(self.img)

  def __getitem__(self, idx):
    label = self.img.iloc[idx, 1]
    img_path = self.dir+f'/{self.labels[label]}/'+self.img.iloc[idx, 0]

    image = Image.open(img_path).convert('RGB')

    if self.transform:
      image = self.transform(image)
    if self.is_test:
      return image

    return (image, label)




In [ ]:
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize((0.5), (0.5))])

In [ ]:
train_data = df[:int(0.8*len(df))]
test_data = df[int(0.8*len(df)):]

In [ ]:
train_data.head()

In [ ]:
train_data = CustomDataset(dataset_folder, train_data, transform=transform)
test_data = CustomDataset(dataset_folder, test_data, transform=transform, is_test=True)

In [ ]:
batch_size = 16

In [ ]:
train_loader = DataLoader(train_data, batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size, shuffle=True, num_workers=2)

In [ ]:
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224',ignore_mismatched_sizes = True)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
model.classifier = nn.Linear(model.config.hidden_size, 3)

In [ ]:
for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model.to(device)

In [ ]:
criterian = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)

In [ ]:
epochs = 10

for epoch in range(epochs):
  model.train()
  training_loss = 0.0
  for i in train_loader:
    image, label = i[0], i[1]
    optimizer.zero_grad()
    output = model(image)
    loss = criterian(output.logits, label)
    loss.backward()
    optimizer.step()
    training_loss += loss.item()
  print(f"epoch: {epoch+1}/10, loss: {training_loss:.2f}")

In [ ]:
model.eval()

In [ ]:
predictions = []

In [ ]:
with torch.no_grad():
  for i in test_loader:
    image = i.to(device)
    output = model(image)
    _, predicted = torch.max(output.logits, 1)
    predictions.extend(predicted.tolist())

In [ ]:
accuracy = accuracy_score(list(test_data.img['labels']), predictions)

In [ ]:
accuracy